# Day 18 — Stockout and Overstock Risk

Create a baseline risk classification using estimated target stock and an assumed inventory position.

**Important:** This is a scenario framework, not a real stock-risk prediction, because actual on-hand inventory and unit demand are unavailable.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

input_path = DATA_DIR / "day17_order_quantity.csv"
if not input_path.exists():
    raise FileNotFoundError(f"Missing {input_path}. Run notebook 15 first.")
df = pd.read_csv(input_path)
required = ["Store","Average_Daily_Sales","Safety_Stock","Reorder_Point","Target_Stock"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")
display(df.head())


,Store,Average_Daily_Sales,Daily_Sales_Std,Observed_Days,Lead_Time_Demand,Safety_Stock,Reorder_Point,Review_Period_Demand,Target_Stock,Assumed_Inventory_Position,Recommended_Order_Quantity
0,1,4759.096031,1012.106393,781,33313.672215,4404.951088,37718.623303,33313.672215,71032.295518,0,71032.295518
1,2,4953.900510,1610.149102,784,34677.303571,7007.788990,41685.092562,34677.303571,76362.396133,0,76362.396133
2,3,6942.568678,2193.383804,779,48597.980745,9546.178585,58144.159329,48597.980745,106742.140074,0,106742.140074
3,4,9638.401786,1936.031881,784,67468.812500,8426.115869,75894.928369,67468.812500,143363.740869,0,143363.740869
4,5,4676.274711,1765.745628,779,32733.922978,7684.985669,40418.908647,32733.922978,73152.831625,0,73152.831625


## Scenario assumptions
Change the placeholder to actual inventory position when data is available.

In [2]:
ASSUMED_INVENTORY_POSITION = 0
df["Scenario_Inventory_Position"] = ASSUMED_INVENTORY_POSITION
df["Days_of_Coverage_Proxy"] = np.where(
    df["Average_Daily_Sales"] > 0,
    df["Scenario_Inventory_Position"] / df["Average_Daily_Sales"],
    np.nan
)
df["Stockout_Risk"] = np.select(
    [
        df["Scenario_Inventory_Position"] <= df["Reorder_Point"],
        df["Scenario_Inventory_Position"] <= df["Target_Stock"]
    ],
    ["High — at/below reorder point", "Moderate — below target stock"],
    default="Lower — at/above target stock"
)
df["Overstock_Scenario"] = np.where(
    df["Scenario_Inventory_Position"] > 1.5 * df["Target_Stock"],
    "Potential overstock",
    "No overstock signal in this scenario"
)
display(df[["Store","Scenario_Inventory_Position","Reorder_Point","Target_Stock","Days_of_Coverage_Proxy","Stockout_Risk","Overstock_Scenario"]].head(10))


,Store,Scenario_Inventory_Position,Reorder_Point,Target_Stock,Days_of_Coverage_Proxy,Stockout_Risk,Overstock_Scenario
0,1,0,37718.623303,71032.295518,0.0,High — at/below reorder point,No overstock signal in this scenario
1,2,0,41685.092562,76362.396133,0.0,High — at/below reorder point,No overstock signal in this scenario
2,3,0,58144.159329,106742.140074,0.0,High — at/below reorder point,No overstock signal in this scenario
3,4,0,75894.928369,143363.740869,0.0,High — at/below reorder point,No overstock signal in this scenario
4,5,0,40418.908647,73152.831625,0.0,High — at/below reorder point,No overstock signal in this scenario
5,6,0,45434.792965,84004.416042,0.0,High — at/below reorder point,No overstock signal in this scenario
6,7,0,72376.384478,134095.740712,0.0,High — at/below reorder point,No overstock signal in this scenario
7,8,0,47019.102627,85794.611555,0.0,High — at/below reorder point,No overstock signal in this scenario
8,9,0,53024.690159,98961.053445,0.0,High — at/below reorder point,No overstock signal in this scenario
9,10,0,43553.750509,82532.696937,0.0,High — at/below reorder point,No overstock signal in this scenario


In [3]:
risk_summary = df["Stockout_Risk"].value_counts(dropna=False).rename_axis("Stockout_Risk").reset_index(name="Store_Count")
display(risk_summary)
df.to_csv(DATA_DIR / "day18_inventory_risk.csv", index=False)
risk_summary.to_csv(DATA_DIR / "day18_inventory_risk_summary.csv", index=False)
print("Saved Day 18 risk outputs.")


,Stockout_Risk,Store_Count
0,High — at/below reorder point,1115


Saved Day 18 risk outputs.


## Conclusion
- Created a scenario-based stockout-risk classification using reorder point and target stock.
- Added an overstock scenario flag and days-of-coverage proxy.
- Outputs: `day18_inventory_risk.csv` and `day18_inventory_risk_summary.csv`.
- Classifications are illustrative only until actual stock records and product-level unit demand are available.